# 03 — Generate only from verified evidence

**10–15 minute lab.** Use the real `GenerationPipeline` to bind an atomic claim to an exact source quote, verify the claim, and cite only evidence used by the final answer.

**Flow:** retrieved source → exact quote + claim → verification → fact ID → verified answer unit.

In [ ]:
import json

from raglab import Citation, ProvenanceStatus
from raglab.errors import GenerationError
from raglab.generation import (
    GenerationConfig,
    GenerationPipeline,
    GenerationRequest,
    ModelInvocation,
)
from raglab.retrieval import (
    CollectionMetadata,
    RankingTrace,
    RetrievalRequest,
    RetrievalResponse,
    RetrievalResult,
)


class StaticRetrieval:
    def __init__(self, results):
        self.results = results

    def collection_metadata(self, collection):
        return CollectionMetadata(collection, "hermetic-embedding", 2)

    def retrieve(self, request):
        return RetrievalResponse(
            request.query, None, (request.query,), request.filters, self.results
        )


class HermeticVerifier:
    def verify(self, pairs):
        return tuple("90 days" not in claim for _, claim in pairs)


class EvidenceModel:
    def __init__(self, facts):
        self.facts = facts
        self.calls = []

    def generate(self, prompt, *, system, schema, config):
        self.calls.append((prompt, system, schema))
        if "select evidence" in system:
            return ModelInvocation({"facts": self.facts}, 80, 18)
        encoded = prompt.split("Verified relevant facts:\n", 1)[1].split(
            "\n\nAnswer only this request:", 1
        )[0]
        rows = json.loads(encoded)
        return ModelInvocation({
            "selected_fact_ids": [rows[0]["fact_id"]],
            "unused_fact_ids": [row["fact_id"] for row in rows[1:]],
        }, 60, 14)

## Checkpoint 1 — Objective: create grounded evidence

**Run:** construct the same public retrieval contract used by the production pipeline.


In [ ]:
query = "What does fault E17 mean?"
citation = Citation(
    source_uri="memory://aster-manual",
    source_name="aster-manual.md",
    title="Aster Greenhouse Controller",
    heading_path=("Fault E17",),
    start_page=None,
    end_page=None,
    start_line=42,
    end_line=45,
    provenance_status=ProvenanceStatus.COMPLETE,
)
evidence = RetrievalResult(
    id="e17-evidence",
    document_id="aster-manual",
    content="Fault E17 means irrigation flow stayed below the safe threshold.",
    citation=citation,
    matched_chunk_ids=("chunk-7",),
    first_chunk_index=7,
    last_chunk_index=7,
    trace=RankingTrace(1, 1, 0.05, 9.2, 0.032, None, None),
)
print("QUERY")
print(f"  {query}")
print("\nRETRIEVAL EVIDENCE")
print(f"  Rank 1 | Source: {evidence.citation.source_name}")
print(f"    Section: {' > '.join(evidence.citation.heading_path)}")
print(f"    Location: lines {evidence.citation.start_line}-{evidence.citation.end_line}")
print(f"    Evidence: {evidence.content}")

### What to observe

Expect full evidence text plus traceable source lines. The model does not receive an anonymous
snippet; it receives evidence that can support a citation.

### Conclusion

Grounding starts before generation: retrieval must preserve both content and provenance.


## Checkpoint 2 — Objective: verify an evidence-bound answer

**Run:** select one query-relevant atomic claim with its source ID and exact quote, then select its pipeline-assigned fact ID. The pipeline renders the verified claim text without a synthesis paraphrase.

In [ ]:
request = GenerationRequest(
    RetrievalRequest(query, collection="lab"),
    GenerationConfig(minimum_sources=1),
)
claim = "Fault E17 means irrigation flow stayed below the safe threshold."
model = EvidenceModel(
    [{"source_id": "S1", "claim": claim, "evidence_quote": evidence.content}],
)
pipeline = GenerationPipeline(
    StaticRetrieval((evidence,)),
    model,
    grounding_verifier=HermeticVerifier(),
    embedding_model="hermetic-embedding",
    embedding_dimension=2,
)
response = pipeline.generate(request)
print("QUERY")
print(f"  {query}")
print("\nRETRIEVAL EVIDENCE")
print(
    f"  Source: {evidence.citation.source_name}, "
    f"lines {evidence.citation.start_line}-{evidence.citation.end_line}"
)
print(f"  Evidence: {evidence.content}")
print("\nSELECTED EVIDENCE")
print(f"  Accepted after quote + NLI validation: {response.metrics.facts_accepted} | {claim}")
print("\nREJECTED FACTS")
print(f"  Invalid quotes: {response.metrics.facts_invalid_quotes} | None")
print(f"  NLI rejected: {response.metrics.facts_nli_rejected} | None")
print("\nGENERATED ANSWER")
print(f"  {response.answer}")
print("\nCITATIONS")
for source in response.sources:
    print(
        f"  {source.id} | {source.citation.source_name} | "
        f"lines {source.citation.start_line}-{source.citation.end_line}"
    )
print("\nSTAGE DIAGNOSIS")
print("  Generation: ANSWERED — the claim is supported by the retrieved evidence.")

### What to observe

Expect one selected, accepted, and used fact plus `S1`. The model proposes text and a fact reference; the pipeline owns `F1`, verifies the final unit against the original quote, and derives the citation from used lineage only.

### Conclusion

A real quote proves provenance, NLI verifies support, and the used fact ID connects the answer to its citation.

## Checkpoint 3 — Objective: reject unsupported evidence and abstain

**Run:** pair an invented password policy with a real but irrelevant quote. The verifier rejects it, leaving no valid evidence for synthesis.

In [ ]:
unsupported_query = "What is the controller password policy?"
invented = "Passwords must be rotated every 90 days."
unsupported_model = EvidenceModel(
    [{"source_id": "S1", "claim": invented, "evidence_quote": evidence.content}],
)
unsupported_pipeline = GenerationPipeline(
    StaticRetrieval((evidence,)),
    unsupported_model,
    grounding_verifier=HermeticVerifier(),
    embedding_model="hermetic-embedding",
    embedding_dimension=2,
)
unsupported = unsupported_pipeline.generate(
    GenerationRequest(
        RetrievalRequest(unsupported_query, collection="lab"),
        GenerationConfig(minimum_sources=1),
    )
)
print("QUERY")
print(f"  {unsupported_query}")
print("\nRETRIEVAL EVIDENCE")
print(f"  Source: {evidence.citation.source_name}")
print(f"  Evidence: {evidence.content}")
print("\nSELECTED EVIDENCE")
print(f"  Accepted after quote + NLI validation: {unsupported.metrics.facts_accepted} | None")
print("\nREJECTED FACTS")
print(f"  Invalid quotes: {unsupported.metrics.facts_invalid_quotes} | None")
print(f"  NLI rejected: {unsupported.metrics.facts_nli_rejected} | {invented}")
print("  Reason: the irrigation evidence does not support a password-rotation claim.")
print("\nGENERATED ANSWER")
print(f"  {unsupported.answer}")
print("\nCITATIONS")
print("  None — rejected facts cannot produce citations.")
print("\nSTAGE DIAGNOSIS")
print("  Retrieval: EVIDENCE FOUND, BUT IRRELEVANT TO THE QUERY")
print("  Generation: ABSTAINED — every proposed fact was rejected as unsupported.")
assert unsupported.abstained and unsupported.source_ids == ()

try:
    unsupported_pipeline.generate(
        GenerationRequest(
            RetrievalRequest(query, collection="lab"),
            GenerationConfig(minimum_sources=1, num_ctx=900, num_predict=128),
        )
    )
except GenerationError as error:
    print("\nSAFETY BOUNDARY")
    print(f"  {error}")

### What to observe

Expect one NLI-rejected selected fact, canonical abstention, no citations, and no synthesis call. An exact quote is NOT enough when it does not entail the claim.

### Conclusion

Unsupported claims fail closed without discarding valid claims from other sources. If every claim is rejected, generation abstains.

## Optional appendix — pinned local verification

Install `raglab[generation]`, download the documented pinned checkpoint, then use `raglab-generate`. Production has no unverified fallback; a missing checkpoint is an actionable error.